# Programação Orientada a Objetos em Python

**Aula completa | Fundamentos, modelagem, reutilização e boas práticas**

Nesta aula vamos aprender a representar conceitos do domínio de um programa usando objetos que reúnem **estado** (dados) e **comportamento** (operações). A sequência parte da sintaxe básica e chega a propriedades, composição, herança, polimorfismo, classes abstratas e `dataclasses`.

## Objetivos

Ao final, você deverá conseguir:
- explicar a diferença entre classe, objeto e instância;
- criar classes com atributos, métodos e inicialização válida;
- proteger regras do domínio com validação e propriedades;
- distinguir métodos de instância, de classe e estáticos;
- escolher entre composição e herança e reconhecer polimorfismo;
- usar classes abstratas e `dataclasses` quando fizer sentido;
- testar invariantes e evitar armadilhas comuns, como atributos mutáveis compartilhados.

**Como estudar:** leia cada seção, execute as células Python na ordem e altere os exemplos para observar como o estado dos objetos muda. O conteúdo foi elaborado a partir das referências oficiais listadas no final; exemplos e explicações foram escritos para esta aula.

## 1. O que é orientação a objetos?

Um programa pode ser organizado em funções que transformam dados. Na orientação a objetos, também agrupamos dados relacionados e as operações válidas sobre eles numa unidade: o **objeto**. Isso ajuda a manter juntas as regras que protegem o estado do objeto e permite criar vários objetos independentes a partir de uma mesma definição.

| Conceito | Significado | Exemplo de domínio |
| --- | --- | --- |
| Classe | Definição de um tipo: quais dados e operações existem | `ContaBancaria` |
| Objeto / instância | Um exemplar concreto criado a partir da classe | conta de Ana, conta de Rui |
| Atributo | Dado associado a uma classe ou instância | titular, saldo |
| Método | Função definida na classe que descreve uma operação | depositar, sacar |
| Estado | Valores atuais dos atributos de um objeto | saldo atual da conta |

Uma analogia: a classe é como uma planta de construção; os objetos são as casas construídas a partir dela. Compartilham uma estrutura prevista, mas cada casa tem valores e estado próprios. Em Python, praticamente todos os valores são objetos; classes definidas por nós criam novos tipos de objeto.

## 2. A primeira classe: classe, instância e atributos

A palavra-chave `class` define uma classe. Chamar o nome da classe como uma função cria uma instância. Cada instância pode armazenar dados próprios. Por convenção, nomes de classes usam `PascalCase` e nomes de métodos/atributos usam `snake_case`.

A célula seguinte cria dois objetos `Produto`. `id(produto_a) != id(produto_b)` indica que são objetos distintos; não use `id()` para lógica de negócio, pois ele é voltado à identidade durante a vida do objeto.

In [1]:
class Produto:
    """Representa um produto simples vendido em uma loja."""

    def __init__(self, nome, preco):
        self.nome = nome
        self.preco = preco

    def descricao(self):
        return f"{self.nome}: R$ {self.preco:.2f}"


produto_a = Produto("Caderno", 18.50)
produto_b = Produto("Caneta", 4.25)

print(produto_a.descricao())
print(produto_b.descricao())
assert produto_a.nome == "Caderno"
assert produto_a is not produto_b

Caderno: R$ 18.50
Caneta: R$ 4.25


## 3. `__init__`, `self` e métodos de instância

`__init__` é executado automaticamente para **inicializar** uma instância recém-criada. Tecnicamente, a criação ocorre antes, em `__new__`; em classes comuns, quase sempre precisamos definir apenas `__init__`. O primeiro parâmetro dos métodos de instância é convencionalmente chamado `self`: Python passa nele a instância que recebeu a chamada.

Assim, `aluna.apresentar()` equivale conceitualmente a `Estudante.apresentar(aluna)`. Não passamos `self` manualmente na chamada usual. `self` não é uma palavra reservada, mas renomeá-lo dificulta a leitura para outras pessoas.

Métodos permitem que o objeto modifique ou consulte o próprio estado. É recomendável inicializar em `__init__` todos os atributos necessários: desse modo, uma instância recém-criada já nasce pronta para ser usada.

In [2]:
class Estudante:
    """Armazena notas de uma pessoa estudante e calcula seu resultado."""

    def __init__(self, nome, notas=None):
        self.nome = nome
        self.notas = [] if notas is None else list(notas)

    def adicionar_nota(self, nota):
        if not 0 <= nota <= 10:
            raise ValueError("A nota deve estar entre 0 e 10.")
        self.notas.append(nota)

    def calcular_media(self):
        if not self.notas:
            return 0.0
        return sum(self.notas) / len(self.notas)

    def situacao(self):
        resultado = "aprovado(a)" if self.calcular_media() >= 7 else "em recuperação"
        return f"{self.nome}: {resultado} (média {self.calcular_media():.1f})"


estudante = Estudante("Lia", [8.0, 9.0])
estudante.adicionar_nota(7.0)
print(estudante.situacao())
assert estudante.calcular_media() == 8.0

try:
    estudante.adicionar_nota(11)
except ValueError as erro:
    print(f"Validação funcionando: {erro}")

Lia: aprovado(a) (média 8.0)
Validação funcionando: A nota deve estar entre 0 e 10.


## 4. Atributos de instância e atributos de classe

Atributos escritos como `self.nome` pertencem a cada instância: `produto_a.preco` pode ser diferente de `produto_b.preco`. Um atributo definido diretamente no corpo da classe pertence à classe e pode ser compartilhado, como uma constante ou uma configuração comum.

**Cuidado com valores mutáveis:** uma lista definida no corpo da classe é uma única lista compartilhada por todas as instâncias que não a sobrescrevam. Para guardar dados independentes, crie a lista dentro de `__init__`. O exemplo mostra primeiro o comportamento compartilhado e depois o desenho correto.

In [5]:
class CarrinhoComErro:
    itens = []  # Uma única lista da classe: compartilhada por todas as instâncias.

    def adicionar(self, item):
        self.itens.append(item)


primeiro_com_erro = CarrinhoComErro()
segundo_com_erro = CarrinhoComErro()
primeiro_com_erro.adicionar("livro")
print("A mesma lista foi alterada:", segundo_com_erro.itens)
assert segundo_com_erro.itens == ["livro"]


class Carrinho:
    total_criados = 0  # Estado compartilhado: contador da classe.

    def __init__(self):
        self.itens = []  # Cada carrinho recebe sua própria lista.
        Carrinho.total_criados += 1

    def adicionar(self, item):
        self.itens.append(item)


primeiro = Carrinho()
segundo = Carrinho()
primeiro.adicionar("livro")
print("Carrinho 1:", primeiro.itens)
print("Carrinho 2:", segundo.itens)
print("Carrinhos criados:", Carrinho.total_criados)
assert primeiro.itens == ["livro"] and segundo.itens == []

A mesma lista foi alterada: ['livro']
Carrinho 1: ['livro']
Carrinho 2: []
Carrinhos criados: 2


## 5. Encapsulamento, convenções e propriedades

**Encapsular** é organizar os detalhes internos e oferecer operações claras para interagir com o objeto. Isso não significa que Python tenha campos privados invioláveis: um nome iniciado por `_`, como `_saldo`, sinaliza por convenção que é detalhe interno e não faz parte da API pública. Um nome iniciado por `__` sofre *name mangling* para reduzir colisões com subclasses; não é uma barreira de segurança.

Uma **invariante** é uma condição que deve permanecer verdadeira enquanto o objeto existe. Por exemplo, o saldo de uma conta não deve ser negativo em determinado modelo. `@property` permite validar uma atribuição usando a aparência de atributo (`conta.saldo = ...`) em vez de chamar um método com sintaxe explícita. Use propriedades para acesso simples e mantenha operações do domínio, como sacar e depositar, como métodos.

In [10]:
class ContaBancaria:
    """Conta com saldo não negativo e operações validadas."""

    def __init__(self, titular, saldo_inicial=0):
        self.titular = titular
        self.saldo = saldo_inicial

    @property
    def saldo(self):
        """Saldo atual, disponível apenas para leitura direta."""
        return self._saldo

    @saldo.setter
    def saldo(self, valor):
        if valor < 0:
            raise ValueError("O saldo inicial não pode ser negativo.")
        self._saldo = valor

    def depositar(self, valor):
        if valor <= 0:
            raise ValueError("O depósito deve ser maior que zero.")
        self._saldo += valor

    def sacar(self, valor):
        if valor <= 0:
            raise ValueError("O saque deve ser maior que zero.")
        if valor > self._saldo:
            raise ValueError("Saldo insuficiente.")
        self._saldo -= valor


conta = ContaBancaria("Ana", 100)
conta.depositar(50)
conta.sacar(30)
print(f"Saldo de {conta.titular}: R$ {conta.saldo:.2f}")
assert conta.saldo == 120

try:
    conta.saldo = -1
except ValueError as erro:
    print(f"Invariante preservada: {erro}")

Saldo de Ana: R$ 120.00
Invariante preservada: O saldo inicial não pode ser negativo.


## 6. Três tipos de método e representações de objetos

- **Método de instância**: recebe `self`; trabalha com o estado daquela instância. É o tipo mais comum.
- **Método de classe**, marcado com `@classmethod`: recebe `cls`, a própria classe. É útil para construtores alternativos, pois respeita subclasses ao instanciá-las.
- **Método estático**, marcado com `@staticmethod`: não recebe `self` nem `cls` automaticamente. Agrupa uma função relacionada conceitualmente à classe, mas que não precisa acessar o objeto nem a classe.

`__repr__` define uma representação informativa, útil para depuração; `__str__` define uma apresentação mais amigável para pessoas. `print(objeto)` usa `__str__`; quando ela não existe, pode recorrer a `__repr__`. Esses métodos especiais são chamados pelo Python em operações conhecidas e devem retornar texto.

In [11]:
class Temperatura:
    """Temperatura internamente armazenada em Celsius."""

    escala_padrao = "Celsius"

    def __init__(self, celsius):
        self.celsius = celsius

    @classmethod
    def de_fahrenheit(cls, fahrenheit):
        celsius = (fahrenheit - 32) * 5 / 9
        return cls(celsius)

    @staticmethod
    def celsius_para_fahrenheit(celsius):
        return celsius * 9 / 5 + 32

    def __repr__(self):
        return f"Temperatura(celsius={self.celsius!r})"

    def __str__(self):
        return f"{self.celsius:.1f} °C"


temperatura = Temperatura.de_fahrenheit(68)
print(temperatura)
print(repr(temperatura))
print("Conversão:", Temperatura.celsius_para_fahrenheit(20), "°F")
assert temperatura.celsius == 20
assert Temperatura.escala_padrao == "Celsius"

20.0 °C
Temperatura(celsius=20.0)
Conversão: 68.0 °F


## 7. Composição: objetos que colaboram

**Composição** descreve uma relação “tem um”: um objeto guarda ou utiliza outro objeto para realizar parte de seu trabalho. Por exemplo, um pedido tem itens; cada item referencia um produto. Essa estratégia permite combinar comportamentos sem formar uma hierarquia de subclasses.

Em geral, prefira composição quando a relação entre os conceitos for “tem um” ou “usa um”. A classe que coordena não precisa recriar as responsabilidades de seus colaboradores; ela delega a operação. Assim, mudanças numa parte do sistema tendem a exigir menos alterações nas demais.

In [12]:
class ItemPedido:
    def __init__(self, produto, quantidade):
        if quantidade <= 0:
            raise ValueError("A quantidade deve ser positiva.")
        self.produto = produto
        self.quantidade = quantidade

    def subtotal(self):
        return self.produto.preco * self.quantidade


class Pedido:
    def __init__(self, numero):
        self.numero = numero
        self.itens = []

    def adicionar_item(self, produto, quantidade):
        self.itens.append(ItemPedido(produto, quantidade))

    def total(self):
        return sum(item.subtotal() for item in self.itens)


produto_livro = Produto("Livro de Python", 80.00)
pedido = Pedido("PED-001")
pedido.adicionar_item(produto_livro, 2)
pedido.adicionar_item(Produto("Marcador", 5.50), 3)
print(f"Pedido {pedido.numero}: R$ {pedido.total():.2f}")
assert pedido.total() == 176.5

Pedido PED-001: R$ 176.50


## 8. Herança e sobrescrita

**Herança** descreve uma relação “é um”: uma classe mais específica pode herdar atributos e métodos de uma classe mais geral. A classe derivada pode **sobrescrever** um método para especializar o comportamento. `super()` delega para a implementação seguinte na ordem de resolução de métodos, sendo a forma idiomática de aproveitar a inicialização e o comportamento herdados.

Herança reduz duplicação quando a relação “é um” é real e estável, mas não deve ser usada apenas para compartilhar algumas linhas de código. Se a subclasse não puder ser usada onde se espera a classe-base sem surpresas, a hierarquia provavelmente está mal modelada; composição pode ser melhor. Python permite herança múltipla, mas para os fundamentos vamos usar herança simples.

In [13]:
class Funcionario:
    def __init__(self, nome, salario):
        if salario < 0:
            raise ValueError("O salário não pode ser negativo.")
        self.nome = nome
        self.salario = salario

    def calcular_bonus(self):
        return self.salario * 0.05


class Gerente(Funcionario):
    def __init__(self, nome, salario, equipe):
        super().__init__(nome, salario)
        self.equipe = equipe

    def calcular_bonus(self):
        return super().calcular_bonus() + self.salario * 0.05


funcionario = Funcionario("Caio", 3000)
gerente = Gerente("Maya", 6000, 4)
print(funcionario.calcular_bonus())
print(gerente.calcular_bonus())
assert isinstance(gerente, Funcionario)
assert gerente.calcular_bonus() == 600

150.0
600.0


## 9. Polimorfismo e classes abstratas

**Polimorfismo** significa que um mesmo trecho de código pode trabalhar com objetos de tipos diferentes, desde que eles ofereçam as operações necessárias. Em Python, muitas vezes basta seguir o protocolo esperado (*duck typing*): “se o objeto se comporta como o necessário, podemos usá-lo”. O chamador depende do comportamento, não precisa testar o tipo concreto de cada objeto.

Quando queremos declarar formalmente um contrato e impedir a criação de uma classe incompleta, podemos usar uma **classe base abstrata** (*ABC*). `@abstractmethod` marca as operações que uma subclasse concreta precisa implementar. A ABC organiza o contrato; o polimorfismo aparece quando chamamos as mesmas operações em instâncias concretas diferentes.

In [14]:
from abc import ABC, abstractmethod


class Forma(ABC):
    @abstractmethod
    def area(self):
        """Calcula a área da forma."""
        raise NotImplementedError


class Retangulo(Forma):
    def __init__(self, largura, altura):
        if largura <= 0 or altura <= 0:
            raise ValueError("As dimensões devem ser positivas.")
        self.largura = largura
        self.altura = altura

    def area(self):
        return self.largura * self.altura


class Circulo(Forma):
    def __init__(self, raio):
        if raio <= 0:
            raise ValueError("O raio deve ser positivo.")
        self.raio = raio

    def area(self):
        return 3.141592653589793 * self.raio**2


def exibir_areas(formas):
    for forma in formas:
        print(f"{type(forma).__name__}: área = {forma.area():.2f}")


formas = [Retangulo(3, 4), Circulo(2)]
exibir_areas(formas)
assert [forma.area() for forma in formas] == [12, 12.566370614359172]

Retangulo: área = 12.00
Circulo: área = 12.57


## 10. `dataclasses`: quando a classe é principalmente dados

Para objetos que representam dados e têm pouca lógica personalizada, o módulo padrão `dataclasses` evita código repetitivo. Com `@dataclass`, anotações de tipo definem campos e Python pode gerar `__init__`, `__repr__` e igualdade (`__eq__`). As anotações ajudam a documentar e a ferramentas de análise estática; por si só, não validam em tempo de execução que os valores recebidos tenham aquele tipo.

Para campos mutáveis, como listas, use `field(default_factory=list)`: a fábrica cria uma lista nova para cada instância. `frozen=True` impede reatribuir campos normalmente; isso imita imutabilidade, mas não torna profundamente imutáveis objetos mutáveis guardados dentro dos campos.

In [15]:
from dataclasses import dataclass, field


@dataclass
class Projeto:
    nome: str
    tecnologias: list[str] = field(default_factory=list)

    def adicionar_tecnologia(self, tecnologia: str) -> None:
        self.tecnologias.append(tecnologia)


projeto_a = Projeto("Sistema acadêmico")
projeto_b = Projeto("Aplicativo de biblioteca")
projeto_a.adicionar_tecnologia("Python")

print(projeto_a)
print(projeto_b)
print("São iguais?", projeto_a == projeto_b)
assert projeto_a.tecnologias == ["Python"]
assert projeto_b.tecnologias == []

Projeto(nome='Sistema acadêmico', tecnologias=['Python'])
Projeto(nome='Aplicativo de biblioteca', tecnologias=[])
São iguais? False


## 11. Métodos especiais: integração com a linguagem

Métodos com nomes como `__init__`, `__str__` e `__repr__` são chamados de métodos especiais (ou *dunder*, de *double underscore*). O interpretador os utiliza para dar significado a operações comuns. Já vimos `__str__` e `__repr__`; outros exemplos incluem `__len__` para `len(objeto)` e `__eq__` para `objeto_a == objeto_b`.

Não implemente métodos especiais apenas por curiosidade: escolha os que expressam naturalmente o significado do tipo. Uma classe que representa uma coleção, por exemplo, pode se beneficiar de `__len__` e `__iter__`; um objeto de domínio normalmente não precisa simular todos os operadores de Python.

In [16]:
class ListaDeTarefas:
    def __init__(self, tarefas=None):
        self._tarefas = list(tarefas) if tarefas is not None else []

    def adicionar(self, tarefa):
        self._tarefas.append(tarefa)

    def __len__(self):
        return len(self._tarefas)

    def __iter__(self):
        return iter(self._tarefas)


tarefas = ListaDeTarefas(["Estudar classes", "Resolver exercício"])
print("Quantidade:", len(tarefas))
for tarefa in tarefas:
    print("-", tarefa)
assert len(tarefas) == 2

Quantidade: 2
- Estudar classes
- Resolver exercício


## 12. Como pensar uma boa modelagem

1. **Comece pelo domínio**, não por uma hierarquia: quais conceitos, dados e regras o problema realmente possui?
2. **Dê responsabilidades coesas**: uma classe deve ter um propósito compreensível; evite classes que fazem tudo.
3. **Proteja invariantes** em construtores e operações. Rejeite dados inválidos cedo, com exceções apropriadas, como `ValueError`.
4. **Prefira interfaces pequenas**: exponha operações necessárias, não todos os detalhes internos.
5. **Use composição primeiro** quando objetos colaboram ou uma entidade contém outra; use herança para uma relação “é um” que faça sentido.
6. **Reduza estado desnecessário**: uma função simples pode ser melhor que uma classe se não há estado nem identidade para representar.
7. **Teste comportamento**, incluindo limites e falhas: objetos recém-criados, operações válidas, entradas inválidas e independência entre instâncias.

POO é uma ferramenta de organização, não uma obrigação para cada pedaço de código. A melhor abstração costuma ser a menor que torna as regras fáceis de entender e manter.

## 13. Exercícios

Tente resolver antes de abrir a célula de respostas.

### Exercício 1 — Retângulo
Crie `Retangulo` com largura e altura positivos. Implemente `area()` e `perimetro()`. Teste com largura 5 e altura 3; os resultados esperados são 15 e 16.

### Exercício 2 — Biblioteca
Crie uma classe `Livro` com título e autor. Crie uma classe `Biblioteca` que mantenha uma coleção própria de livros. Implemente `adicionar_livro(livro)` e `buscar_por_titulo(titulo)`. Adicionar um livro a uma biblioteca não deve alterar outra.

### Exercício 3 — Formas polimórficas
Amplie a classe abstrata `Forma` com uma nova forma geométrica. Inclua-a na lista já usada por `exibir_areas` sem mudar a função. O que isso demonstra sobre polimorfismo?

### Exercício 4 — Inspeção
Crie duas instâncias da mesma classe com listas mutáveis. Demonstre que alterar a lista de uma não modifica a outra. Explique por que uma lista definida diretamente no corpo da classe não atende a esse requisito.

### Desafio — Pagamentos
Projete um contrato `MetodoPagamento` com um método abstrato `pagar(valor)` e duas implementações concretas. Escreva uma função que receba qualquer objeto que ofereça `pagar` e a utilize sem verificar o tipo concreto.

In [17]:
class RetanguloExercicio:
    def __init__(self, largura, altura):
        if largura <= 0 or altura <= 0:
            raise ValueError("Largura e altura devem ser positivas.")
        self.largura = largura
        self.altura = altura

    def area(self):
        return self.largura * self.altura

    def perimetro(self):
        return 2 * (self.largura + self.altura)


retangulo_exercicio = RetanguloExercicio(5, 3)
assert retangulo_exercicio.area() == 15
assert retangulo_exercicio.perimetro() == 16
print("Exercício 1:", retangulo_exercicio.area(), retangulo_exercicio.perimetro())

Exercício 1: 15 16


## 14. Síntese

- **Classe** define estrutura e comportamento; **instância** é um objeto concreto.
- `__init__` prepara o estado; `self` identifica a instância que recebeu a chamada.
- Atributos de instância guardam estado individual; atributos de classe guardam dados compartilhados intencionalmente.
- Métodos e propriedades devem preservar as regras do objeto; `_nome` é uma convenção de detalhe interno, não privacidade absoluta.
- **Composição** modela colaboração (“tem um”); **herança** modela especialização (“é um”).
- **Polimorfismo** permite usar uma mesma interface com implementações diferentes.
- `@dataclass` reduz boilerplate para classes predominantemente voltadas a dados.
- Uma classe só é útil se tornar o modelo ou as regras mais claros; funções simples continuam sendo uma ótima escolha quando bastam.

## Referências consultadas

As explicações desta aula foram verificadas com a documentação oficial do Python. A documentação é mantida pelo projeto Python e pode ser consultada para aprofundar cada tema:

- [The Python Tutorial — Classes](https://docs.python.org/3/tutorial/classes.html): definição de classes, instâncias, métodos, variáveis de classe e de instância, herança e convenção para atributos não públicos.
- [Python Language Reference — Data Model](https://docs.python.org/3/reference/datamodel.html): identidade, tipo e valor dos objetos; propriedades, instâncias e métodos especiais.
- [The Python Standard Library — `dataclasses`](https://docs.python.org/3/library/dataclasses.html): campos, valores padrão, `default_factory`, igualdade, `frozen` e herança em classes de dados.
- [The Python Standard Library — `abc`](https://docs.python.org/3/library/abc.html): classes base abstratas e `@abstractmethod`.
- [The Python Tutorial — Errors and Exceptions](https://docs.python.org/3/tutorial/errors.html): exceções usadas para comunicar entradas e operações inválidas.

Os exemplos desta aula são originais e independentes de pacotes externos; basta um kernel Python.